# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² Mass Spectrometry EVs dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library, with all fields referenced by their `@id` where possible.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is a Metadata object
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

First, enumerate all record sets and their fields. All references will use the `@id` from the Croissant schema.

In [ ]:
# List all record sets (`cr:RecordSet`) and their fields using the @id for reference
record_sets = metadata.find_all('@type', 'cr:RecordSet')

print("Available Record Sets:")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")

print("\nFields for each RecordSet:")
for rs in record_sets:
    print(f"Fields for RecordSet @id={rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            field_obj = metadata.find_by_id(field['@id'])
            print(f"    - @id: {field_obj['@id']}, name: {field_obj.get('name', '(no name)')}, dataType: {field_obj.get('dataType', '')}")
    else:
        print("    (No fields found)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Refer to record set and field `@id`s from the previous cell.

In [ ]:
# Prepare to extract data using record set @id's. List all record sets to choose from.
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet @id={record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

if dataframes:
    # Just pick the first available
    first_rs_id = next(iter(dataframes.keys()))
    print(f"\nColumns in DataFrame for @id={first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record set dataframes extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section uses `@id` references exclusively (where possible), and demonstrates filtering/normalization on numeric fields.

In [ ]:
# Choose a representative RecordSet and numeric/categorical fields by `@id`.
# For this example, we use the first loaded record set, and try to infer numeric fields.
import numpy as np

if dataframes:
    record_set_id = first_rs_id
    df = dataframes[record_set_id]
    # Infer numeric columns (float/int)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field (by @id): {numeric_field_id}")
    else:
        print("No numeric columns found.")
        numeric_field_id = None

    # Filtering on the numeric column (if any)
    threshold = df[numeric_field_id].mean() if numeric_field_id else None
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a suitable group field (categorical)
        group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by field (by @id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No categorical fields for grouping found.")
    else:
        print("No numeric field for analysis.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions or relationships of the data. (Example: Histogram of the numeric field and boxplot by group.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group (if group field exists)
    if group_fields:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² Mass Spectrometry EVs dataset using the `mlcroissant` library. 

- All dataset loading and field selection was conducted via Croissant `@id` identifiers for standards compliance.
- We explored all available record sets, fields, and provided sample visualizations and filtering operations.

**Key findings, data quirks, or next steps for domain-specific analysis should be summarized here.**